# Quick Validation

This notebook is only for a fast sanity check of generated CSV files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

processed_dir = Path('../data/processed')
files = {
    path.stem: pd.read_csv(path)
    for path in sorted(processed_dir.glob('*.csv'))
}
sorted(files)


In [ ]:
for name, df in files.items():
    print(f'\n{name}: shape={df.shape}')
    display(df.head(10))


In [ ]:
summary = {
    'n_sites': len(files['sites']) if 'sites' in files else 0,
    'n_profiles': len(files['profiles']) if 'profiles' in files else 0,
    'n_shoreline_observations': len(files['shoreline_observations']) if 'shoreline_observations' in files else 0,
    'n_intervals': len(files['interval_metrics']) if 'interval_metrics' in files else 0,
}
pd.Series(summary, name='count').to_frame()


In [ ]:
if 'interval_metrics' in files:
    interval_df = files['interval_metrics'].copy()
    interval_df['date_start'] = pd.to_datetime(interval_df['date_start'])
    interval_df['year'] = interval_df['date_start'].dt.year
    (
        interval_df.groupby('year')['retreat_rate_m_per_year']
        .mean()
        .plot(marker='o', figsize=(9, 4), title='Mean Retreat Rate By Interval Start Year')
    )
    plt.ylabel('m/year')
    plt.show()

if 'shoreline_observations' in files:
    obs_df = files['shoreline_observations'].copy()
    obs_df['survey_year'] = pd.to_numeric(obs_df['survey_year'], errors='coerce')
    (
        obs_df.groupby('site_id')['obs_id']
        .count()
        .sort_values(ascending=False)
        .plot(kind='bar', figsize=(10, 4), title='Observation Count By Site')
    )
    plt.ylabel('rows')
    plt.show()

if 'wind_obs_hourly' in files:
    wind_df = files['wind_obs_hourly'].copy()
    wind_df['year'] = pd.to_numeric(wind_df['year'], errors='coerce')
    (
        wind_df.groupby('year')['wind_speed_ms']
        .mean()
        .dropna()
        .plot(marker='o', figsize=(9, 4), title='Mean Wind Speed By Year')
    )
    plt.ylabel('m/s')
    plt.show()
